In [1]:
# Imports
!pip install -q ultralytics==8.3.221 torchmetrics seaborn opencv-python-headless pandas

import os
import json, time, shutil
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import torchvision.transforms as T


# Config
@dataclass
class CFG:
    model_type: str = "yolo"                # "faster_rcnn" or "yolo"

    ckpt: str = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints/yolo_selected_best.pt"
    #ckpt: str = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints/faster_rcnn_best.pth"

    protocol_dir: str = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints"

    test_images_dir: str = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/images"
    test_labels_dir: str = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/labels"

    # Inference
    batch_size: int = 4
    num_workers: int = 2
    input_size: int = 640
    num_classes: int = 3

    # Thresholds
    score_low: float = 0.001      # mAP floor
    score_prf: float = 0.30       # locked confidence from val
    iou_thresh: float = 0.50

    # YOLO-specific postprocessing
    nms_iou: float = 0.7
    max_det: int = 100

    # FP analysis
    run_fp_analysis: bool = True
    fp_topk: int = 500

    out_dir: str = "/kaggle/working/evaluation"


CFG = CFG()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

for p in [CFG.ckpt, CFG.test_images_dir, CFG.test_labels_dir, CFG.protocol_dir]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Not found: {p}")

n_imgs = len([f for f in os.listdir(CFG.test_images_dir)
              if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))])
print(f"Test images on disk: {n_imgs}")


# 0=bg, 1=smoke, 2=fire
CLASS_NAMES = {1: "smoke", 2: "fire"}


# Dataset (returns PIL for YOLO path, tensor for Faster R-CNN path)
class FireSmokeDataset(torch.utils.data.Dataset):
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def __init__(self, images_dir, labels_dir, transforms=None, return_pil=False):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.transforms = transforms
        self.return_pil = return_pil
        self.files = sorted(p for p in self.images_dir.iterdir()
                            if p.suffix.lower() in self.IMG_EXTS)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []
        lbl = self.labels_dir / (img_path.stem + ".txt")
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, bw, bh = map(float, parts)
                x1 = max(0.0, (xc - bw / 2) * w)
                y1 = max(0.0, (yc - bh / 2) * h)
                x2 = min(w, (xc + bw / 2) * w)
                y2 = min(h, (yc + bh / 2) * h)
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(1 if int(cls) == 0 else 2)   # 0→1 smoke, 1→2 fire

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if not self.return_pil and self.transforms:
            img = self.transforms(img)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
                    if boxes.numel() else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.zeros((boxes.shape[0],), dtype=torch.int64),
        }
        return img, target, str(img_path)


def collate_fn(batch):
    return tuple(zip(*batch))


# Build loader depending on model_type
is_yolo = (CFG.model_type == "yolo")

test_ds = FireSmokeDataset(
    CFG.test_images_dir,
    CFG.test_labels_dir,
    transforms=None if is_yolo else T.ToTensor(),
    return_pil=is_yolo,
)
print(f"Test dataset: {len(test_ds)} images "
      f"(return_pil={test_ds.return_pil})")

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_fn,
    pin_memory=True,
)



# Models and inference
ckpt_epoch = "N/A"
model_name = "Unknown"


# Faster R-CNN
if CFG.model_type == "faster_rcnn":
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection import fasterrcnn_resnet50_fpn

    def load_faster_rcnn(ckpt_path, num_classes, input_size, device):
        model = fasterrcnn_resnet50_fpn(weights=None)
        in_f = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
        model.transform.min_size = (input_size,)
        model.transform.max_size = input_size
        ckpt = torch.load(ckpt_path, map_location="cpu")
        model.load_state_dict(ckpt["model_state_dict"])
        model.to(device).eval()
        return model, ckpt.get("epoch", "?")

    model, ckpt_epoch = load_faster_rcnn(
        CFG.ckpt, CFG.num_classes, CFG.input_size, DEVICE
    )
    model_name = "Faster R-CNN ResNet-50 FPN"
    print(f"Loaded {model_name} (epoch {ckpt_epoch})")

    @torch.no_grad()
    def run_inference(model, loader, device, score_thresh):
        preds, targets, records = [], [], []
        n_img = 0
        t0 = time.time()
        for images, tgts, paths in tqdm(loader, desc="Inference"):
            images = [im.to(device, non_blocking=True) for im in images]
            outputs = model(images)
            for out, tgt, p in zip(outputs, tgts, paths):
                keep = out["scores"] >= score_thresh
                preds.append({
                    "boxes": out["boxes"][keep].cpu(),
                    "scores": out["scores"][keep].cpu(),
                    "labels": out["labels"][keep].cpu(),
                })
                targets.append({
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu(),
                })
                records.append({
                    "path": p,
                    "boxes": out["boxes"][keep].cpu().numpy(),
                    "scores": out["scores"][keep].cpu().numpy(),
                    "labels": out["labels"][keep].cpu().numpy(),
                    "gt_boxes": tgt["boxes"].cpu().numpy(),
                    "gt_labels": tgt["labels"].cpu().numpy(),
                })
            n_img += len(images)
        return preds, targets, records, n_img / max(time.time() - t0, 1e-6)

    preds, targets, records, fps = run_inference(
        model, test_loader, DEVICE, CFG.score_low
    )


# YOLOv8s (via baseline-input adapter)
elif CFG.model_type == "yolo":
    import sys

    required = ["final_yolo_protocol.py", "budget_protocol.py"]
    for name in required:
        f = Path(CFG.protocol_dir) / name
        if not f.is_file():
            raise FileNotFoundError(
                f"Missing {name} in {CFG.protocol_dir}. "
                f"final_yolo_protocol imports budget_protocol, both must be present."
            )

    if CFG.protocol_dir not in sys.path:
        sys.path.insert(0, CFG.protocol_dir)

    from final_yolo_protocol import predict_with_baseline_input
    from ultralytics import YOLO

    yolo_wrapper = YOLO(CFG.ckpt)
    core = yolo_wrapper.model.to(DEVICE).float().eval()
    model_name = "YOLOv8s (baseline-input adapter)"
    ckpt_epoch = 17   # from README_YOLO.md — selected_best.pt = epoch 17

    print(f"Loaded {model_name} from {CFG.ckpt}")
    print(f"classes: {yolo_wrapper.names}, nc={yolo_wrapper.model.nc}")
    assert yolo_wrapper.model.nc == 2, f"expected 2 classes, got {yolo_wrapper.model.nc}"
    assert dict(yolo_wrapper.names) == {0: "smoke", 1: "fire"}, yolo_wrapper.names

    @torch.no_grad()
    def run_inference_yolo(core, loader, device, score_floor, nms_iou, max_det):
        preds, targets, records = [], [], []
        n_img = 0
        t0 = time.time()

        for images, tgts, paths in tqdm(loader, desc="YOLO inference"):
            # images — list[PIL.Image] (return_pil=True in Dataset)
            out = predict_with_baseline_input(
                core, list(images), device=device,
                score_floor=score_floor,
                nms_iou=nms_iou,
                max_det=max_det,
            )
            for o, tgt, p in zip(out, tgts, paths):
                boxes = o["boxes"].cpu()
                scores = o["scores"].cpu()
                labels = (o["labels"].cpu() + 1).long()   # 0/1 → 1/2

                preds.append({"boxes": boxes, "scores": scores, "labels": labels})
                targets.append({
                    "boxes": tgt["boxes"].cpu(),
                    "labels": tgt["labels"].cpu(),
                })
                records.append({
                    "path": p,
                    "boxes": boxes.numpy(),
                    "scores": scores.numpy(),
                    "labels": labels.numpy(),
                    "gt_boxes": tgt["boxes"].numpy(),
                    "gt_labels": tgt["labels"].numpy(),
                })
            n_img += len(images)

        return preds, targets, records, n_img / max(time.time() - t0, 1e-6)

    preds, targets, records, fps = run_inference_yolo(
        core, test_loader, DEVICE,
        score_floor=CFG.score_low,
        nms_iou=CFG.nms_iou,
        max_det=CFG.max_det,
    )

else:
    raise ValueError(f"Unknown model_type in CFG: {CFG.model_type}")

print(f"Inference FPS: {fps:.2f}")


# Sanity check: IoU between pred and GT should be non-trivial
print("\n=== Sanity check: per-GT max IoU (first 3 non-empty images) ===")
found = 0
for rec in records:
    if len(rec["boxes"]) > 0 and len(rec["gt_boxes"]) > 0:
        iou = box_iou(torch.tensor(rec["gt_boxes"]), torch.tensor(rec["boxes"]))
        print(f"{Path(rec['path']).name}: "
              f"max IoU per GT = {iou.max(dim=1).values.numpy()[:6]}")
        found += 1
        if found >= 3:
            break
if found == 0:
    print("No images with both GT and pred — something is wrong.")
elif found < 3:
    print(f"Only {found} such images — inspect above.")
else:
    print("If you see at least some values >= 0.5 → pipeline is correct.")



# Metrics: mAP
def compute_map(preds, targets):
    metric = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox", class_metrics=True
    )
    p_remap = [
        {"boxes": p["boxes"], "scores": p["scores"], "labels": p["labels"] - 1}
        for p in preds
    ]
    t_remap = [
        {"boxes": t["boxes"], "labels": t["labels"] - 1}
        for t in targets
    ]
    metric.update(p_remap, t_remap)
    return metric.compute()


m = compute_map(preds, targets)
mAP50 = float(m["map_50"])
mAP5095 = float(m["map"])
per_cls = [float(x) for x in m["map_per_class"]]   # order = [0=smoke, 1=fire]

print(f"\nmAP@0.5      = {mAP50:.4f}")
print(f"mAP@0.5:0.95 = {mAP5095:.4f}")
if len(per_cls) >= 2:
    print(f"mAP@0.5:0.95 [smoke, fire] = {per_cls[0]:.4f}, {per_cls[1]:.4f}")


# Per-class P/R/F1
def per_class_prf(preds, targets, iou_thresh, score_thresh):
    stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    for p, t in zip(preds, targets):
        for cls in [1, 2]:
            pm = (p["labels"] == cls) & (p["scores"] >= score_thresh)
            pb, tb = p["boxes"][pm], t["boxes"][t["labels"] == cls]
            if len(tb) == 0:
                stats[cls]["fp"] += len(pb)
                continue
            if len(pb) == 0:
                stats[cls]["fn"] += len(tb)
                continue
            ious, matched = box_iou(pb, tb), set()
            for i in range(len(pb)):
                bi, bj = 0.0, -1
                for j in range(len(tb)):
                    if j in matched:
                        continue
                    v = ious[i, j].item()
                    if v > bi:
                        bi, bj = v, j
                if bi >= iou_thresh and bj >= 0:
                    stats[cls]["tp"] += 1
                    matched.add(bj)
                else:
                    stats[cls]["fp"] += 1
            stats[cls]["fn"] += len(tb) - len(matched)

    out = {}
    for cls, s in stats.items():
        tp, fp, fn = s["tp"], s["fp"], s["fn"]
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        out[CLASS_NAMES[cls]] = {
            "precision": round(prec, 4),
            "recall": round(rec, 4),
            "f1": round(f1, 4),
            "tp": tp, "fp": fp, "fn": fn,
        }
    return out


prf = per_class_prf(preds, targets, CFG.iou_thresh, CFG.score_prf)
print("\nPer-class P/R/F1:")
print(json.dumps(prf, indent=2))


# Confusion matrix
def confusion_matrix_counts(preds, targets, iou_thresh, score_thresh):
    cm = np.zeros((3, 3), dtype=int)   # rows=GT bg/smoke/fire, cols=Pred bg/smoke/fire
    for p, t in zip(preds, targets):
        matched = set()
        for i in range(len(p["boxes"])):
            if p["scores"][i] < score_thresh:
                continue
            pc = int(p["labels"][i])
            bi, bj = 0.0, -1
            for j in range(len(t["boxes"])):
                if j in matched:
                    continue
                v = box_iou(p["boxes"][i:i+1], t["boxes"][j:j+1]).item()
                if v > bi:
                    bi, bj = v, j
            if bi >= iou_thresh and bj >= 0:
                cm[int(t["labels"][bj]), pc] += 1
                matched.add(bj)
            else:
                cm[0, pc] += 1
        for j in range(len(t["boxes"])):
            if j not in matched:
                cm[int(t["labels"][j]), 0] += 1
    return cm


cm = confusion_matrix_counts(preds, targets, CFG.iou_thresh, CFG.score_prf)
print("\nConfusion matrix (rows=GT bg/smoke/fire, cols=Pred bg/smoke/fire):")
print(cm)


# threshold sweep & empty-image FP rate
empty_idx = [i for i, r in enumerate(records) if len(r["gt_boxes"]) == 0]
print(f"Empty (no-GT) test images: {len(empty_idx)} / {len(records)}")

for thresh in [0.10, 0.30, 0.50, 0.70]:
    fp_empty = sum(int((records[i]["scores"] >= thresh).sum()) for i in empty_idx)
    # total FP across all images (not just empty): FP = preds with no GT match
    total_fp = 0
    for rec in records:
        for i in range(len(rec["boxes"])):
            if rec["scores"][i] < thresh:
                continue
            if len(rec["gt_boxes"]) == 0:
                total_fp += 1
                continue
            ious = box_iou(torch.tensor(rec["boxes"][i:i+1]),
                           torch.tensor(rec["gt_boxes"]))
            if ious.max().item() < CFG.iou_thresh:
                total_fp += 1
    print(f"  conf>={thresh:.2f}: FP(all)={total_fp}, "
          f"FP/empty={fp_empty/len(empty_idx):.3f}, "
          f"FP/img={total_fp/len(records):.3f}")


# Figures
fig_dir = Path(CFG.out_dir) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["bg", "smoke", "fire"],
    yticklabels=["bg", "smoke", "fire"],
)
plt.xlabel("Predicted")
plt.ylabel("Ground truth")
plt.title(f"Confusion matrix @ IoU={CFG.iou_thresh}, score={CFG.score_prf}")
plt.tight_layout()
plt.savefig(fig_dir / "confusion_matrix.png", dpi=150)
plt.close()

cls_names = list(prf.keys())
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(cls_names))
w = 0.25
for i, metric in enumerate(["precision", "recall", "f1"]):
    ax.bar(x + (i - 1) * w, [prf[c][metric] for c in cls_names], w, label=metric)
ax.set_xticks(x)
ax.set_xticklabels(cls_names)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.legend()
ax.set_title(f"Per-class P/R/F1 @ IoU={CFG.iou_thresh}, score={CFG.score_prf}")
plt.tight_layout()
plt.savefig(fig_dir / "per_class_prf.png", dpi=150)
plt.close()
print(f"\nFigures → {fig_dir}")



# False-positive candidates
df_fp = None
if CFG.run_fp_analysis:
    import cv2
    import pandas as pd

    fp_dir = Path(CFG.out_dir) / "fp_analysis"
    crops_dir = fp_dir / "crops"
    crops_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for rec in records:
        for i in range(len(rec["boxes"])):
            if rec["scores"][i] < CFG.score_prf:
                continue
            pb = rec["boxes"][i]
            matched = False
            if len(rec["gt_boxes"]) > 0:
                ious = box_iou(
                    torch.tensor(pb[None]), torch.tensor(rec["gt_boxes"])
                )
                if ious.max().item() >= CFG.iou_thresh:
                    matched = True
            if matched:
                continue
            x1, y1, x2, y2 = map(int, pb)
            img = cv2.imread(rec["path"])
            if img is None:
                continue
            h, w = img.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            fname = f"{Path(rec['path']).stem}_fp{i}.jpg"
            cv2.imwrite(str(crops_dir / fname), crop)
            rows.append({
                "image": rec["path"],
                "crop": fname,
                "pred_class": CLASS_NAMES.get(int(rec["labels"][i]), "?"),
                "score": float(rec["scores"][i]),
                "category": "",
                "notes": "",
            })

    df_fp = (
        pd.DataFrame(rows)
        .sort_values("score", ascending=False)
        .head(CFG.fp_topk)
    )
    df_fp.to_csv(fp_dir / "fp_candidates.csv", index=False)
    print(f"Saved {len(df_fp)} FP candidates → {fp_dir / 'fp_candidates.csv'}")



# Save metrics JSON
results = {
    "model": model_name,
    "model_type": CFG.model_type,
    "checkpoint": CFG.ckpt,
    "checkpoint_epoch": ckpt_epoch,
    "split": "test",
    "test_images_dir": CFG.test_images_dir,
    "num_images": len(test_ds),
    "mAP@0.5": mAP50,
    "mAP@0.5:0.95": mAP5095,
    "per_class_mAP@0.5:0.95": {
        "smoke": per_cls[0] if len(per_cls) > 0 else None,
        "fire":  per_cls[1] if len(per_cls) > 1 else None,
    },
    "per_class_PRF@0.5": prf,
    "confusion_matrix": cm.tolist(),
    "fps": fps,
    "iou_thresh": CFG.iou_thresh,
    "score_thresh_prf": CFG.score_prf,
    "score_thresh_inference": CFG.score_low,
}

res_filename = f"{CFG.model_type}_test_thr{CFG.score_prf:.2f}.json"
res_path = Path(CFG.out_dir) / "results" / res_filename
res_path.parent.mkdir(parents=True, exist_ok=True)
res_path.write_text(json.dumps(results, indent=2))
print(f"\nMetrics JSON → {res_path}")



# Final summary
print(f"\nFINAL SUMMARY — {model_name}")
print(f"Checkpoint epoch : {ckpt_epoch}")
print(f"Test images      : {len(test_ds)}")
print(f"mAP@0.5          : {mAP50:.4f}")
print(f"mAP@0.5:0.95     : {mAP5095:.4f}")
for c in ["smoke", "fire"]:
    p = prf[c]
    print(f"  {c:6s} P/R/F1   : "
          f"{p['precision']:.3f} / {p['recall']:.3f} / {p['f1']:.3f}   "
          f"(tp={p['tp']}, fp={p['fp']}, fn={p['fn']})")
print(f"FPS (T4)         : {fps:.2f}")
if df_fp is not None:
    print(f"FP candidates    : {len(df_fp)}")
print("=" * 60 + f"\nArtifacts → {CFG.out_dir}")

archive_path = f"/kaggle/working/evaluation_{CFG.model_type}"
shutil.make_archive(archive_path, "zip", CFG.out_dir)
print(f"ZIP → {archive_path}.zip")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.9 MB/s eta 0:00:0000:01
Device: cuda
GPU: Tesla T4
Test images on disk: 4306
Test dataset: 4306 images (return_pil=False)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 177MB/s] 


Loaded Faster R-CNN ResNet-50 FPN (epoch 6)


Inference:   0%|          | 0/1077 [00:00<?, ?it/s]

Inference FPS: 5.79

=== Sanity check: per-GT max IoU (first 3 non-empty images) ===
AoF06978.jpg: max IoU per GT = [0.8022579]
AoF07618.jpg: max IoU per GT = [0.6922259]
AoF07673.jpg: max IoU per GT = [0.5205146]
If you see at least some values >= 0.5 → pipeline is correct.

mAP@0.5      = 0.7687
mAP@0.5:0.95 = 0.4112
mAP@0.5:0.95 [smoke, fire] = 0.4700, 0.3523

Per-class P/R/F1:
{
  "smoke": {
    "precision": 0.758,
    "recall": 0.8228,
    "f1": 0.7891,
    "tp": 1895,
    "fp": 605,
    "fn": 408
  },
  "fire": {
    "precision": 0.6176,
    "recall": 0.7224,
    "f1": 0.6659,
    "tp": 2079,
    "fp": 1287,
    "fn": 799
  }
}

Confusion matrix (rows=GT bg/smoke/fire, cols=Pred bg/smoke/fire):
[[   0  594 1277]
 [ 399 1894   10]
 [ 787   12 2079]]
Empty (no-GT) test images: 2008 / 4306
  conf>=0.10: FP(all)=15717, FP/empty=2.685, FP/img=3.650
  conf>=0.30: FP(all)=4751, FP/empty=0.128, FP/img=1.103
  conf>=0.50: FP(all)=3000, FP/empty=0.078, FP/img=0.697
  conf>=0.70: FP(all)=18